In [1]:
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-03-20 14:33:55--  https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400395283 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.0-bin-hadoop3.tgz’

spark-3.5.0-bin-had 100%[===================>] 381.85M  18.1MB/s    in 22s     

2026-03-20 14:34:17 (17.8 MB/s) - ‘spark-3.5.0-bin-hadoop3.tgz’ saved [400395283/400395283]



In [16]:
import os
import findspark

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Lab2_StackOverflow") \
    .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.17.0") \
    .getOrCreate()

In [17]:
languages_df = spark.read.csv("programming-languages.csv", header=True)
languages_list = [row['name'].lower() for row in languages_df.collect()]

In [18]:
from pyspark.sql import functions as F

posts_raw = spark.read \
    .format("xml") \
    .option("rowTag", "row") \
    .load("posts_sample.xml")

posts_df = posts_raw.select(
    F.col("_CreationDate").alias("date"),
    F.col("_Tags").alias("tags")
).filter("tags IS NOT NULL")

In [19]:
processed_posts = posts_df \
    .withColumn("year", F.year(F.col("date"))) \
    .filter("year >= 2010 AND year <= 2020") \
    .withColumn("tag", F.explode(F.split(F.regexp_replace(F.col("tags"), r"^<|>$", ""), "><"))) \
    .withColumn("tag", F.lower(F.col("tag")))

In [20]:
from pyspark.sql.window import Window

final_counts = processed_posts \
    .filter(F.col("tag").isin(languages_list)) \
    .groupBy("year", "tag") \
    .agg(F.count("tag").alias("count"))

window_spec = Window.partitionBy("year").orderBy(F.desc("count"))

top_10_report = final_counts \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter("rank <= 10") \
    .orderBy("year", "rank")

In [21]:
!rm -rf top_10_languages.parquet
top_10_report.write.parquet("top_10_languages.parquet")
spark.read.parquet("top_10_languages.parquet").show(100)

+----+-----------+-----+----+
|year|        tag|count|rank|
+----+-----------+-----+----+
|2010|       java|   52|   1|
|2010|        php|   46|   2|
|2010| javascript|   44|   3|
|2010|     python|   26|   4|
|2010|objective-c|   23|   5|
|2010|          c|   20|   6|
|2010|       ruby|   12|   7|
|2010|     delphi|    8|   8|
|2010|applescript|    3|   9|
|2010|          r|    3|  10|
|2011|        php|  102|   1|
|2011|       java|   93|   2|
|2011| javascript|   83|   3|
|2011|     python|   37|   4|
|2011|objective-c|   34|   5|
|2011|          c|   24|   6|
|2011|       ruby|   20|   7|
|2011|       perl|    9|   8|
|2011|     delphi|    8|   9|
|2011|       bash|    7|  10|
|2012|        php|  154|   1|
|2012| javascript|  132|   2|
|2012|       java|  124|   3|
|2012|     python|   69|   4|
|2012|objective-c|   45|   5|
|2012|       ruby|   27|   6|
|2012|          c|   27|   7|
|2012|       bash|   10|   8|
|2012|          r|    9|   9|
|2012|      scala|    6|  10|
|2013|    